<a href="https://colab.research.google.com/github/aspurser84-dot/BIFX546-project/blob/main/notebooks/Supplemental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Title: Supplemental: COVID-19 Variant tracking in California between 2021 through 2023

##Author: Amanda Graham

###Course: BIFX 546 — Machine Learning for Bioinformatics, Spring 2026

###Instructor: Dr. Sarangan (Ravi) Ravichandran

Date: 5/6/26

Disclaimer: This notebook was created as a supplemental notebook to the Final_project_BIFX546_Graham. This supplemental code uses alternative train/ test/ split methods that did not result in the final model but are important to show that they were tested. This notebook uses a publically available dataset, key concepts from BIFX 546 Machine Learning for Bioinformatics and concepts from the textbook Data Sciencefrom Scratch.

#Setting up directory

The below code changes the working directory so the data can be properly saved in Colab.

In [1]:
import os

# Show current working directory
print("Old directory:", os.getcwd())

# Move up to main directory
os.chdir("/")

print("New directory:", os.getcwd())

Old directory: /content
New directory: /


#Upload the data

Import the dataset from Kagglehub repository. This requires an API key setup in CoLab.

Note: The main notebook shows two methods to upload the dataset. For simplicity, this supplemental notebook uses the direct download method from the Kagglehub repository.

In [2]:
#Imports the dataset from Kagglehub repository and
#places the dataset in kaggle/input/ folder
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nidzsharma/covid-19-variant-data")

# Verify the actual data file is present, if not force re-download
csv_file = os.path.join(path, "covid19_variant.csv")

if not os.path.exists(csv_file):
  path = kagglehub.dataset_download("nidzsharma/covid-19-variant-data", force_download=True)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'covid-19-variant-data' dataset.
Path to dataset files: /kaggle/input/covid-19-variant-data


#Data Cleaning

The main notbook provides an overview of exploratory data analysis (EDA) with explanations on why each data cleaning step is preformed. See the main notebook for detailed explanations.

The below code converts the downloaded csv file to a pandas dataframe, converts the date formate to datetime data type, removes the variables 'area' and 'area_type' and removes the rows with the name 'Total' and 'Other' found in the 'variant_name' column.

In [3]:
#Converts csv file to pandas dataframe
import pandas as pd

df = pd.read_csv(csv_file)

#converts 'date' variable to datetime data type
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')

#Removes variables 'area' and 'area_type'
df1 = df.copy()
df1 = df1.drop(columns=['area', 'area_type'])

#Remove all rows with "Total" in variant_name
A_value_to_remove = 'Total'
B_value_to_remove = 'Other'

#Remove rows where 'variant_name' equals "Total" and "Other"
df1 = df1[df1['variant_name'] != A_value_to_remove]
df1 = df1[df1['variant_name'] != B_value_to_remove]

The below code calculates a 7 day average for the variables specimens and percentage. Then all N/A values are filled in with zero. The key step is to sort the data by variant name and date so the average can be properly calculated.

In [4]:
# Calculate 7-day rolling average for specimens and percentage
#specimens_7d_avg
data_sorted = df1.sort_values(by= ['variant_name',"date"], ascending=True)
series_specimen = pd.Series(data_sorted['specimens'])

moving_avg_specimens = series_specimen.rolling(window=7).mean()

#percentage_7d_avg
data = df1['percentage'].tolist()
series_percentage = pd.Series(data_sorted['percentage'])

moving_avg_percentage = series_percentage.rolling(window=7).mean()

#Fill in N/A values with 7 day average

df2 = df1.copy()
df2['specimens_7d_avg'] = moving_avg_specimens
df2['percentage_7d_avg'] = moving_avg_percentage

df2['specimens_7d_avg'] = df2['specimens_7d_avg'].fillna(0)
df2['percentage_7d_avg'] = df2['percentage_7d_avg'].fillna(0)

The below code calculates the rate of change for the 'specimens' variable, calculates a 7 day average and fills in any N/A values with zero.

In [5]:
#calculates the rate change for number of specimens per day
#Also fills in the N/A values with zero
rate_change = df2.groupby('variant_name', as_index=False)['specimens'].diff()

df3 = df2.copy()
df3['rate_change'] = rate_change

df3['rate_change'] = df3['rate_change'].fillna(0)

# Calculate 7-day rolling average
#rate change

data_sorted2 = df3.sort_values(by= ['variant_name',"date"], ascending=True)
series_rate = pd.Series(data_sorted2['rate_change'])

moving_avg_rate = series_rate.rolling(window=7).mean()

df3['rate_change_7d_avg'] = moving_avg_rate

df3['rate_change_7d_avg'] = df3['rate_change_7d_avg'].fillna(0)

The below code removes any rows where the were no specimens collected.

In [6]:
# Remove rows where there were no specimens collected
df4 = df3.copy()

# Value to drop from a specific column
threshold = 1

# Drop rows where the column matches the value
df5 = df4[df4['specimens'] >= threshold].reset_index(drop=True)

print("The original dataset contained", df4.shape, "rows and columns.")
print("The new dataset contains", df5.shape, "rows and columns.")

The original dataset contained (6232, 8) rows and columns.
The new dataset contains (1797, 8) rows and columns.


Encode data

In [7]:
#encode catagorical values to numeric
#this is needed to preform scalar fit transform

encoder_list = df5['variant_name'].unique()
encoder_values = [i for i in range(len(encoder_list))]
encoder_dict = dict(zip(encoder_list, encoder_values))
df5['variant_name'] = df5['variant_name'].map(encoder_dict)
pd.Series(df5["variant_name"]).value_counts()
print(encoder_list, encoder_values)

['Alpha' 'Epsilon' 'Omicron' 'Delta' 'Gamma' 'Mu' 'Beta' 'Lambda'] [0, 1, 2, 3, 4, 5, 6, 7]


#Random Forest Modeling

##Time Series Split
The below code splits the dataset using Time series split. This package ensures all testing data comes after the training data.

In [8]:
#Time series split
from sklearn.model_selection import TimeSeriesSplit

df_sorted = df5.sort_values(by='date')
y = df_sorted['variant_name'] # Extract the target variable 'variant_name'
X = df_sorted.drop(columns=['date', 'variant_name']) # Extract features (dropping date and variant_name)

tscv = TimeSeriesSplit(n_splits=2)

for train_idx, test_idx in tscv.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f'X Training set : {X_train.shape[0]} samples')
print(f'X Test set     : {X_test.shape[0]} samples')
print(f'Y Training set : {y_train.shape[0]} samples')
print(f'Y Test set     : {y_test.shape[0]} samples')
print(f'Features     : {X_train.shape[1]}')

feature_names = X_train.columns.tolist()

X Training set : 1198 samples
X Test set     : 599 samples
Y Training set : 1198 samples
Y Test set     : 599 samples
Features     : 6


The below code provides a class distribution for the training dataset. It shows the variants are not evenly represented in the training data with a range from 5.3-20.4% distribution.

In [9]:
#Class distribution
import numpy as np
unique, counts = np.unique(y_train, return_counts=True)
print("Training set class distribution:")
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}: {cnt} samples ({cnt/len(y_train)*100:.1f}%)")

Training set class distribution:
  Class 0: 244 samples (20.4%)
  Class 1: 185 samples (15.4%)
  Class 2: 64 samples (5.3%)
  Class 3: 238 samples (19.9%)
  Class 4: 190 samples (15.9%)
  Class 5: 121 samples (10.1%)
  Class 6: 72 samples (6.0%)
  Class 7: 84 samples (7.0%)


The below code calculates the computed weights per each class. The highest weight is with class 2 which had the lower distribution in the training dataset.

In [10]:
#Show the computed weights
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight('balanced',
                               classes=np.unique(y_train), y=y_train)
print("Computed class weights:")
for cls, w in zip(np.unique(y_train), weights):
    print(f"  Class {cls}: {w:.2f}")
print(f"\nRatio: minority gets {weights[1]/weights[0]:.1f}x more weight than majority")

Computed class weights:
  Class 0: 0.61
  Class 1: 0.81
  Class 2: 2.34
  Class 3: 0.63
  Class 4: 0.79
  Class 5: 1.24
  Class 6: 2.08
  Class 7: 1.78

Ratio: minority gets 1.3x more weight than majority


#Random Forest model
The code below builds a Random Forest model with a balanced class weight and starting with 100 trees. This model has 99% training accuracy but only 11.5% testing accuracy. However, only four of the variants are present in the testing dataset.

In [11]:
#Original Random Forest model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# Create a Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, criterion='entropy', oob_score=True,
                             class_weight='balanced',
                             random_state=42)

# Fit the model
clf.fit(X_train, y_train)

print(f"Training accuracy : {clf.score(X_train, y_train):.0%}")

#OOB, out of bag score, uses out of bag samples to test the model
#Close to testing accuracy but not identical
print(f"OOB score (≈ test): {clf.oob_score_:.2%}")
print("The Random Forest model was trained on the following features:", feature_names)

from sklearn.metrics import classification_report
#Make predictions
y_pred = clf.predict(X_test)

variant_name = encoder_list

print("=== RandomForest ===")
#support number of samples
print(classification_report(y_test, y_pred, labels= encoder_values, target_names= variant_name, zero_division=0 ))

#Accuracy, all predictions correct
accuracy = accuracy_score(y_test, y_pred)
print(f"Testing accuracy: {accuracy:.2%}")

#Precision, true positives correctly called
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Precision: {precision:.2%}")

#Recall, true positive rate
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Recall: {recall:.2%}")

#F1 score, model accuracy balanced
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"F1 Score: {f1:.2%}")

Training accuracy : 99%
OOB score (≈ test): 60.68%
The Random Forest model was trained on the following features: ['specimens', 'percentage', 'specimens_7d_avg', 'percentage_7d_avg', 'rate_change', 'rate_change_7d_avg']
=== RandomForest ===
              precision    recall  f1-score   support

       Alpha       0.00      0.00      0.00         0
     Epsilon       0.04      0.50      0.07         2
     Omicron       0.83      0.04      0.08       470
       Delta       0.10      0.40      0.16       120
       Gamma       0.04      0.14      0.06         7
          Mu       0.00      0.00      0.00         0
        Beta       0.00      0.00      0.00         0
      Lambda       0.00      0.00      0.00         0

    accuracy                           0.12       599
   macro avg       0.13      0.14      0.05       599
weighted avg       0.67      0.12      0.09       599

Testing accuracy: 11.52%
Precision: 66.90%
Recall: 11.52%
F1 Score: 9.38%


#Balanced Random Forest model
The code below builds a Balanced Random Forest model starting with 100 trees. This model has 87% training accuracy but only 11.2% testing accuracy. This model is not better than the original Random Forest model.

In [12]:
#Balanced random forest model
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from imblearn.ensemble import BalancedRandomForestClassifier

variant_name = encoder_list

# BalancedRandomForest: undersamples the majority class for each tree
brf = BalancedRandomForestClassifier(n_estimators=100, random_state=42)
brf.fit(X_train, y_train)
y_pred_brf = brf.predict(X_test)
y_proba_brf = brf.predict_proba(X_test)

print(f"Training accuracy : {brf.score(X_train, y_train):.0%}")

print("=== BalancedRandomForest ===")
print(classification_report(y_test, y_pred_brf, labels= encoder_values, target_names= variant_name, zero_division=0))

#Accuracy, all predictions correct
accuracy = accuracy_score(y_test, y_pred_brf)
print(f"Testing accuracy: {accuracy:.2%}")

#Precision, true positives correctly called
precision = precision_score(y_test, y_pred_brf, average='weighted', zero_division=0)
print(f"Precision: {precision:.2%}")

#Recall, true positive rate
recall = recall_score(y_test, y_pred_brf, average='weighted', zero_division=0)
print(f"Recall: {recall:.2%}")

#F1 score, model accuracy balanced
f1 = f1_score(y_test, y_pred_brf, average='weighted', zero_division=0)
print(f"F1 Score: {f1:.2%}")

Training accuracy : 87%
=== BalancedRandomForest ===
              precision    recall  f1-score   support

       Alpha       0.00      0.00      0.00         0
     Epsilon       0.00      0.00      0.00         2
     Omicron       0.72      0.04      0.08       470
       Delta       0.10      0.38      0.16       120
       Gamma       0.00      0.00      0.00         7
          Mu       0.00      0.00      0.00         0
        Beta       0.00      0.00      0.00         0
      Lambda       0.00      0.00      0.00         0

    accuracy                           0.11       599
   macro avg       0.10      0.05      0.03       599
weighted avg       0.59      0.11      0.10       599

Testing accuracy: 11.19%
Precision: 58.77%
Recall: 11.19%
F1 Score: 9.72%


#Easy Ensemble model
The code below builds a Balanced Random Forest model. This model has 41% training accuracy but only 9.7% testing accuracy. This model is the worst model for this dataset.

In [13]:
# EasyEnsemble: trains multiple classifiers on balanced subsets
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
ee = EasyEnsembleClassifier(n_estimators=10, random_state=42)
ee.fit(X_train, y_train)

variant_name = encoder_list

y_pred_ee = ee.predict(X_test)
y_proba_ee = ee.predict_proba(X_test)[:, 1]

print(f"Training accuracy : {ee.score(X_train, y_train):.0%}")

print("=== EasyEnsembleClassifier ===")
print(classification_report(y_test, y_pred_ee, labels= encoder_values, target_names= variant_name, zero_division=0))

#Accuracy, all predictions correct
accuracy = accuracy_score(y_test, y_pred_ee)
print(f"Testing accuracy: {accuracy:.2%}")

#Precision, true positives correctly called
precision = precision_score(y_test, y_pred_ee, average='weighted', zero_division=0)
print(f"Precision: {precision:.2%}")

#Recall, true positive rate
recall = recall_score(y_test, y_pred_ee, average='weighted', zero_division=0)
print(f"Recall: {recall:.2%}")

#F1 score, model accuracy balanced
f1 = f1_score(y_test, y_pred_ee, average='weighted', zero_division=0)
print(f"F1 Score: {f1:.2%}")

Training accuracy : 41%
=== EasyEnsembleClassifier ===
              precision    recall  f1-score   support

       Alpha       0.00      0.00      0.00         0
     Epsilon       0.00      0.00      0.00         2
     Omicron       0.45      0.03      0.06       470
       Delta       0.09      0.37      0.15       120
       Gamma       0.00      0.00      0.00         7
          Mu       0.00      0.00      0.00         0
        Beta       0.00      0.00      0.00         0
      Lambda       0.00      0.00      0.00         0

    accuracy                           0.10       599
   macro avg       0.07      0.05      0.03       599
weighted avg       0.37      0.10      0.07       599

Testing accuracy: 9.68%
Precision: 37.31%
Recall: 9.68%
F1 Score: 7.38%


#SMOTE = Synthetic Minority Over-sampling Technique
The code below using the existing training dataset and adds additional synthetic points to build a Random Forest model. The testing dataset is kept the same. This model has 99% training accuracy but only 10% testing accuracy. This model is the worst model for this dataset.

In [14]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
print("The original training data contained", X_train.shape, "X training data and", y_train.shape, "y training data")
print("The new training data contains", X_res.shape, "X training data and", y_res.shape, "y training data")

The original training data contained (1198, 6) X training data and (1198,) y training data
The new training data contains (1952, 6) X training data and (1952,) y training data


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# Create a Random Forest Classifier
clf2 = RandomForestClassifier(n_estimators=100, criterion='entropy', oob_score=True,
                             class_weight='balanced',
                             random_state=42)

# Fit the model
clf2.fit(X_res, y_res)

# Make predictions
y_pred2 = clf2.predict(X_test)

variant_name = encoder_list

print(f"Training accuracy : {clf2.score(X_train, y_train):.0%}")

#support number of samples
print(classification_report(y_test, y_pred2, labels= encoder_values, target_names= variant_name, zero_division=0 ))

#Accuracy, all predictions correct
accuracy2 = accuracy_score(y_test, y_pred2)
print(f"Testing accuracy: {accuracy2:.2%}")

#Precision, true positives correctly called
precision2 = precision_score(y_test, y_pred2, average='weighted', zero_division=0)
print(f"Precision: {precision2:.2%}")

#Recall, true positive rate
recall2 = recall_score(y_test, y_pred2, average='weighted', zero_division=0)
print(f"Recall: {recall2:.2%}")

#F1 score, model accuracy balanced
f12 = f1_score(y_test, y_pred2, average='weighted', zero_division=0)
print(f"F1 Score: {f1:.2%}")

Training accuracy : 99%
              precision    recall  f1-score   support

       Alpha       0.00      0.00      0.00         0
     Epsilon       0.00      0.00      0.00         2
     Omicron       0.74      0.03      0.06       470
       Delta       0.10      0.41      0.16       120
       Gamma       0.04      0.14      0.07         7
          Mu       0.00      0.00      0.00         0
        Beta       0.00      0.00      0.00         0
      Lambda       0.00      0.00      0.00         0

    accuracy                           0.11       599
   macro avg       0.11      0.07      0.04       599
weighted avg       0.60      0.11      0.08       599

Testing accuracy: 10.68%
Precision: 59.94%
Recall: 10.68%
F1 Score: 7.38%


#Conclusion
Time series split is not the best way to split this dataset. This creates an imbalanced dataset that only has four variants present in the training dataset.
